# 01 — Data Split

`gold.match_features` is the canonical match-level training table: one row
per match, sides canonicalized by ATP id ordering, with the full balanced
feature set (player + opponent rolling stats, differentials, context) and
`match_won` relative to the canonical player side.

In [1]:
from src.utils import load_env

load_env()

from src.constants import DATA_PROCESSED, GOLD_TABLE, PROFILES_TABLE

gold_table = GOLD_TABLE

profiles_table = PROFILES_TABLE

test_size = 0.2

val_size = 0.2

random_state = 42

cutoff_date = None  # default: computed from data in the split cell; override via papermill

val_cutoff_date = None  # default: computed from data in the split cell; override via papermill

output_dir = str(DATA_PROCESSED)

In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from src.db.training import to_dataframe
from src.features.columns import (
    FEATURE_COLS,
)

Path(output_dir).mkdir(parents=True, exist_ok=True)

In [3]:
print("Loading gold features...")

df = to_dataframe(f"SELECT * FROM {gold_table} ORDER BY match_date")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Date range: {df['match_date'].min()} to {df['match_date'].max()}")

Loading gold features...
Loaded 20 rows, 83 columns
Date range: 2026-01-13 00:00:00 to 2026-08-27 00:00:00


In [4]:
# gold.match_features is the canonical match-level training table: one row
# per match, sides canonicalized by ATP id ordering, full balanced feature
# set, and match_won relative to the canonical player (vs opponent).

matches = df
print(f"Matches: {len(matches)}")

Matches: 20


In [10]:
# Train/validation/test are chronological bands of the data's date range
# (defaults: 60/20/20; override via papermill-supplied val_cutoff_date /
# cutoff_date). The validation band drives Optuna/early stopping/model
# selection; the test band is the final evaluation split and is never
# consumed by tuning, early stopping, or base-model selection.
span = df["match_date"].max() - df["match_date"].min()
cutoff_date = cutoff_date if cutoff_date is not None else df["match_date"].min() + span * 0.8
val_cutoff_date = (
    val_cutoff_date if val_cutoff_date is not None else df["match_date"].min() + span * 0.6
)

train = matches[matches["match_date"] < val_cutoff_date]
val = matches[(matches["match_date"] >= val_cutoff_date) & (matches["match_date"] < cutoff_date)]
test = matches[matches["match_date"] >= cutoff_date]
for _name, _split in (("train", train), ("val", val), ("test", test)):
    assert not _split.empty, f"{_name} split is empty; adjust the chronological cutoffs"

X_train, y_train = train[FEATURE_COLS].copy(), train["match_won"]
X_val, y_val = val[FEATURE_COLS].copy(), val["match_won"]
X_test, y_test = test[FEATURE_COLS].copy(), test["match_won"]

# Every minimal inference input required to rebuild each held-out row:
# match date, canonical ids, surface, tournament/round context, indoor state.
info_cols = [
    "match_id",
    "match_date",
    "player_id",
    "opponent_id",
    "surface",
    "tournament",
    "round",
    "tournament_level",
    "round_encoded",
    "is_indoor",
]

# The finalized gold contract: every FEATURE_COLS cell is already
# non-null and finite (dbt + snapshot validation enforce this). No
# imputer is fitted here; the split asserts the contract so training
# can never read NULL/NaN/Infinity feature values.
for _name, _split in (("train", X_train), ("val", X_val), ("test", X_test)):
    _values = np.asarray(_split, dtype=float)
    assert not np.isnan(_values).any(), f"gold FEATURE_COLS contains NaN in {_name}"
    assert np.isfinite(_values).all(), f"gold FEATURE_COLS non-finite in {_name}"

print(f"Training set: {len(train)} rows")
print(f"Validation set: {len(val)} rows")
print(f"Test set: {len(test)} rows")

Training set: 16 rows
Test set: 4 rows


In [6]:
# ── Save ──
pd.DataFrame({"y": y_train}).to_parquet(f"{output_dir}/y_train.parquet", index=False)
pd.DataFrame({"y": y_val}).to_parquet(f"{output_dir}/y_val.parquet", index=False)
pd.DataFrame({"y": y_test}).to_parquet(f"{output_dir}/y_test.parquet", index=False)

for name, data in [
    ("X_train", X_train),
    ("X_val", X_val),
    ("X_test", X_test),
    ("info_train", train[info_cols]),
    ("info_val", val[info_cols]),
    ("info_test", test[info_cols]),
]:
    path = f"{output_dir}/{name}.parquet"
    data.to_parquet(path)
    print(f"Saved {path} ({len(data)} rows)")

# Max match date across every split (train, validation, test); evaluation
# rejects the candidate before registration when this is later than the
# current UTC date.
split_meta = {
    "max_match_date": str(df["match_date"].max().date()),
    "n_train": int(len(train)),
    "n_val": int(len(val)),
    "n_test": int(len(test)),
    "val_cutoff_date": str(val_cutoff_date.date()),
    "test_cutoff_date": str(cutoff_date.date()),
}
with open(f"{output_dir}/split_meta.json", "w") as f:
    json.dump(split_meta, f, indent=2)
print(f"Max match date across all splits: {split_meta['max_match_date']}")

Saved data/processed/X_train.parquet (16 rows)
Saved data/processed/X_test.parquet (4 rows)
Saved data/processed/info_train.parquet (16 rows)
Saved data/processed/info_test.parquet (4 rows)
